# Demo Interativo — Incerteza por Pixel em Reconstrucao Acelerada de MRI

**Autor**: Massanori
**TCC**: Quantificacao de incerteza por pixel para reconstrucao acelerada de MRI cerebral
**Artefato**: bloco S15 — *demo clicavel* dos 3 grupos calibrados (A/B/C)

Este notebook carrega **1 volume pre-computado** do split `test`, os **3 modulos de incerteza ja calibrados** (S5.7) e, para um slice com lesao:

1. gera os mapas de incerteza dos 3 grupos lado a lado;
2. calcula as metricas daquele slice (Coverage, largura de intervalo, IoU, ULAS) reusando `src/metrics`;
3. monta a **hero figure** (GT | reconstrucao | erro | ResM | QR | QR-Lesion) com a lesao destacada.

| Grupo | Metodo | Calibracao |
|---|---|---|
| **A** | ResM (Residual Magnitude, baseline) | Scaled-CP locally-adaptive |
| **B** | QR (CQR, replicacao de Giannakopoulos et al. 2026) | CQR aditivo |
| **C** | QR-Lesion (contribuicao original, pinball ponderada por lesao, lambda=5) | CQR aditivo |

> **Cobertura nominal**: 90% (`alpha = 0.10`). A garantia de cobertura marginal e formal sob exchangeability (Romano, Patterson & Candes, 2019).

### Como rodar
- **Kaggle** (recomendado hoje): *Settings -> Accelerator -> T4* (opcional; roda em CPU) e *Add Input* com os datasets listados na celula de configuracao. O notebook auto-detecta `/kaggle/input`.
- **Colab / publico**: defina os repositorios HuggingFace na celula de configuracao (ver bloco 3.2). Sem isso, o notebook procura os caminhos locais/Kaggle.


In [ ]:
# Celula 1 — Setup: clona o repo (Colab) ou reusa (Kaggle) + instala dependencias
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/KR0N0S7/tcc-mri-uncertainty.git"

# Em Kaggle o working dir e /kaggle/working; em Colab e /content.
if Path("/kaggle/working").is_dir():
    BASE = Path("/kaggle/working")
elif Path("/content").is_dir():
    BASE = Path("/content")
else:
    BASE = Path.cwd()

REPO_DIR = BASE / "tcc-mri-uncertainty"

# Se ja estamos dentro do repo (rodando localmente), reusa.
_here = Path.cwd()
if (_here / "src" / "models" / "uncertainty_modules.py").is_file():
    REPO_DIR = _here
elif REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repo:", REPO_DIR)
subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=False)

# Dependencias. fastmri traz a U-Net usada pelos 3 modulos.
print("\nInstalando dependencias (fastmri, torch, ...)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(REPO_DIR / "requirements.txt"), "huggingface_hub"],
    check=False,
)
print("Setup OK")


In [ ]:
# Celula 2 — Configuracao das fontes de dados/checkpoints
#
# O notebook resolve as fontes nesta ordem de prioridade:
#   1. HuggingFace Hub  (se HF_MODEL_REPO != None)  -> demo PUBLICO (bloco 3.2)
#   2. Kaggle /kaggle/input (datasets privados anexados) -> como roda hoje
#   3. Caminhos locais  (clone dos datasets em disco)
#
# Datasets Kaggle esperados (Add Input):
#   - tcc-mri-recons-varnet-brain-4x   (split test/, .npz por volume)
#   - tcc-mri-lesion-masks             (mascaras .pt por volume)
#   - tcc-mri-resm-checkpoints         (best.pt do Grupo A)
#   - tcc-mri-qr-checkpoints           (best.pt do Grupo B)
#   - tcc-mri-qr-lesion-checkpoints    (best.pt do Grupo C)
#   - tcc-mri-conformal-qhats          (q_hat_A/B/C.json do S5.7)

from pathlib import Path

# ---- Opcao 1: HuggingFace (preencher apos o bloco 3.2) --------------------
# Ex.: "massanorikishi/tcc-mri-uncertainty" (modelos) e o mesmo/clone p/ amostra.
HF_MODEL_REPO = None       # ex.: "<usuario>/tcc-mri-uncertainty"  (repo_type="model")
HF_SAMPLE_REPO = None      # ex.: "<usuario>/tcc-mri-demo-sample" (repo_type="dataset")

# ---- Selecao do volume/slice ----------------------------------------------
# Deixe VOLUME_ID = None para o notebook escolher automaticamente o slice
# de teste com a MAIOR area de lesao (demo mais ilustrativa).
VOLUME_ID = None           # ex.: "file_brain_AXFLAIR_200_6002460"
SLICE_IDX = None           # ex.: 7  (ignorado se VOLUME_ID is None)

ALPHA = 0.10               # cobertura nominal 90%
TOP_PCT = 0.05             # X em top-X% para o IoU
N_PERMS_ULAS = 10          # permutacoes do null baseline do ULAS

print("HF_MODEL_REPO :", HF_MODEL_REPO)
print("VOLUME_ID     :", VOLUME_ID, "| SLICE_IDX:", SLICE_IDX)


In [ ]:
# Celula 3 — Resolve caminhos de recons, mascaras, checkpoints e q_hats
import json
import torch

RECONS_SLUG = "tcc-mri-recons-varnet-brain-4x"
MASKS_SLUG = "tcc-mri-lesion-masks"
CKPT_SLUGS = {"A": "tcc-mri-resm-checkpoints",
              "B": "tcc-mri-qr-checkpoints",
              "C": "tcc-mri-qr-lesion-checkpoints"}
QHATS_SLUG = "tcc-mri-conformal-qhats"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


def _kaggle_candidates(slug):
    cands = [Path("/kaggle/input") / slug]
    root = Path("/kaggle/input/datasets")
    if root.is_dir():
        cands += [u / slug for u in root.iterdir() if u.is_dir()]
    return [c for c in cands if c.is_dir()]


def _find_in_kaggle(slug, validator):
    for c in _kaggle_candidates(slug):
        if validator(c):
            return c
        # tolera 1 nivel extra de aninhamento
        subs = [p for p in c.iterdir() if p.is_dir()]
        if len(subs) == 1 and validator(subs[0]):
            return subs[0]
    return None


def _find_best_pt(root):
    hits = [p for p in root.rglob("best.pt") if ".ipynb_checkpoints" not in p.parts]
    hits = sorted(hits, key=lambda p: len(p.relative_to(root).parts))
    return hits[0] if hits else None


# --- HuggingFace (demo publico) -------------------------------------------
def _resolve_hf():
    if HF_MODEL_REPO is None:
        return None
    from huggingface_hub import hf_hub_download, snapshot_download
    print("Baixando artefatos do HuggingFace Hub...")
    ckpt = {}
    for g, fname in [("A", "resm/best_state_dict.pt"),
                     ("B", "qr/best_state_dict.pt"),
                     ("C", "qr_lesion/best_state_dict.pt")]:
        ckpt[g] = Path(hf_hub_download(HF_MODEL_REPO, fname, repo_type="model"))
    qhats = {}
    for g in "ABC":
        qhats[g] = Path(hf_hub_download(HF_MODEL_REPO, f"qhats/q_hat_{g}.json",
                                        repo_type="model"))
    sample_repo = HF_SAMPLE_REPO or HF_MODEL_REPO
    sample_dir = Path(snapshot_download(sample_repo, repo_type="dataset"))
    return dict(checkpoints=ckpt, qhats=qhats,
                recons_test=sample_dir / "test", masks=sample_dir / "masks")


# --- Kaggle ----------------------------------------------------------------
def _resolve_kaggle():
    recons = _find_in_kaggle(
        RECONS_SLUG, lambda p: all((p / s).is_dir() for s in ("train", "val", "cal", "test")))
    if recons is None:
        return None
    masks = _find_in_kaggle(MASKS_SLUG, lambda p: any(p.glob("*.pt")))
    qroot = _find_in_kaggle(
        QHATS_SLUG, lambda p: all((p / f"q_hat_{g}.json").is_file() for g in "ABC"))
    ckpt = {}
    for g, slug in CKPT_SLUGS.items():
        for c in _kaggle_candidates(slug):
            b = _find_best_pt(c)
            if b:
                ckpt[g] = b
                break
    if masks is None or qroot is None or len(ckpt) != 3:
        return None
    return dict(checkpoints=ckpt,
                qhats={g: qroot / f"q_hat_{g}.json" for g in "ABC"},
                recons_test=recons / "test", masks=masks)


# --- Local -----------------------------------------------------------------
def _resolve_local():
    data = Path("data")
    recons_test = data / "recons" / "test"
    masks = data / "masks"
    ckdir = Path("checkpoints")
    ckpt = {"A": ckdir / "resm" / "best.pt",
            "B": ckdir / "qr" / "best.pt",
            "C": ckdir / "qr_lesion" / "best.pt"}
    qhats = {g: Path("results") / "qhats" / f"q_hat_{g}.json" for g in "ABC"}
    if recons_test.is_dir() and all(p.is_file() for p in ckpt.values()):
        return dict(checkpoints=ckpt, qhats=qhats, recons_test=recons_test, masks=masks)
    return None


SRC = _resolve_hf() or _resolve_kaggle() or _resolve_local()
if SRC is None:
    raise FileNotFoundError(
        "Nenhuma fonte resolvida. Em Kaggle: anexe os 6 datasets (ver celula 2). "
        "Em Colab: preencha HF_MODEL_REPO/HF_SAMPLE_REPO apos o bloco 3.2."
    )

print("\nFontes resolvidas:")
print("  recons/test:", SRC["recons_test"])
print("  masks      :", SRC["masks"])
for g in "ABC":
    print(f"  ckpt {g}    :", SRC["checkpoints"][g])
    print(f"  q_hat {g}   :", SRC["qhats"][g])


In [ ]:
# Celula 4 — Instancia os 3 modulos, carrega checkpoints e os q_hats calibrados
from src.models import QuantileRegressionModule, ResidualMagnitudeModule

CHANS, POOLS = 32, 4  # mesma arquitetura do treino (S5)


def _load_state_dict_flexible(path, module):
    '''Carrega pesos no modulo, aceitando os dois formatos relevantes:

    - best.pt do treino (S5): dict com a chave 'model_state_dict'
      (ver src/training/train_loop.py: save_checkpoint).
    - state_dict puro: formato publicado no HuggingFace pelo bloco 3.2
      (apenas pesos, sem optimizer/scheduler/config nem dados de paciente).
    '''
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict) and "model_state_dict" in obj:
        sd = obj["model_state_dict"]
    elif isinstance(obj, dict) and "model" in obj and isinstance(obj["model"], dict):
        sd = obj["model"]            # tolera nome legado
    else:
        sd = obj                      # assume state_dict puro
    module.load_state_dict(sd)
    return module


modules, qhats = {}, {}
for g in "ABC":
    mod = (ResidualMagnitudeModule(chans=CHANS, num_pool_layers=POOLS) if g == "A"
           else QuantileRegressionModule(chans=CHANS, num_pool_layers=POOLS))
    _load_state_dict_flexible(SRC["checkpoints"][g], mod)
    modules[g] = mod.to(DEVICE).eval()
    qhats[g] = json.loads(Path(SRC["qhats"][g]).read_text())
    n = sum(p.numel() for p in mod.parameters())
    print(f"Grupo {g}: {n:,} params | metodo={qhats[g]['method']} | "
          f"q_hat={qhats[g]['q_hat']:.6f}")


In [ ]:
# Celula 5 — Monta o dataset do split test e escolhe um slice com lesao
from src.data import ReconsSliceDataset

ds = ReconsSliceDataset(SRC["recons_test"], masks_dir=SRC["masks"])
print(f"Split test: {len(ds)} slices")


def _pick_index():
    # Slice explicito pedido pelo usuario
    if VOLUME_ID is not None and SLICE_IDX is not None:
        for i, (npz, s) in enumerate(ds.index):
            if npz.stem == VOLUME_ID and s == SLICE_IDX:
                return i
        raise ValueError(f"{VOLUME_ID} slice {SLICE_IDX} nao encontrado no test.")
    # Auto: maior area de lesao
    best_i, best_area = None, -1
    for i in range(len(ds)):
        m = ds[i]["lesion_mask"]
        area = int((m > 0.5).sum().item())
        if area > best_area:
            best_area, best_i = area, i
    if best_area <= 0:
        raise RuntimeError("Nenhum slice com lesao no split test (verifique masks).")
    return best_i


idx = _pick_index()
sample = ds[idx]
print(f"\nSlice escolhido: vol={sample['volume_id']} | slice={sample['slice_idx']} "
      f"| seq={sample['sequence']} | pixels de lesao="
      f"{int((sample['lesion_mask'] > 0.5).sum().item())}")

# Tensores (1,1,H,W) no device
recon = sample["recon"].unsqueeze(0).to(DEVICE)
target = sample["target"].unsqueeze(0).to(DEVICE)
error = sample["error_map"].unsqueeze(0)            # |y-x| normalizado (CPU)
lesion = sample["lesion_mask"].unsqueeze(0)          # (1,1,H,W) CPU


In [ ]:
# Celula 6 — Forward dos 3 grupos + intervalo calibrado + mapa de incerteza
from src.calibration import apply_cqr_interval, apply_resm_interval


def predict(group):
    '''Retorna (lower_cal, upper_cal, u_log, halfwidth_cal) em CPU.

    - u_log: a 'incerteza' usada nas metricas, igual ao compute_metrics.py
      (ResM: u(x); QR: (upper-lower)/2 pre-calibracao).
    - halfwidth_cal: (upper_cal - lower_cal)/2, a incerteza POS-calibracao
      efetivamente afirmada pelo metodo — e o que vai na hero figure
      (comparavel entre os 3 grupos).
    '''
    q = float(qhats[group]["q_hat"])
    with torch.no_grad():
        out = modules[group](recon)
    if group == "A":
        u = out
        lo, hi = apply_resm_interval(recon, u, q)
    else:
        lo, hi = apply_cqr_interval(out["lower"], out["upper"], q)
        u = (out["upper"] - out["lower"]) / 2.0
    hw = (hi - lo) / 2.0
    return lo.cpu(), hi.cpu(), u.cpu(), hw.cpu()


pred = {g: predict(g) for g in "ABC"}
print("Forward + calibracao OK para A/B/C.")
for g in "ABC":
    lo, hi, u, hw = pred[g]
    print(f"  {g}: halfwidth medio (cal) = {hw.mean().item():.5f} | "
          f"u_log medio = {u.mean().item():.5f}")


In [ ]:
# Celula 7 — Metricas daquele slice, reusando exatamente src/metrics
from src.metrics.coverage import empirical_coverage, mean_interval_width
from src.metrics.iou import iou_topk
from src.metrics.ulas import ulas_with_null

rows = {}
for g in "ABC":
    lo, hi, u, hw = pred[g]
    cov_g = empirical_coverage(lo, hi, target.cpu())["coverage"]
    cov_l = empirical_coverage(lo, hi, target.cpu(), mask=lesion)["coverage"]
    w_g = mean_interval_width(lo, hi)
    w_l = mean_interval_width(lo, hi, mask=lesion)
    iou_g = iou_topk(u, error, top_pct=TOP_PCT)
    iou_l = iou_topk(u, error, top_pct=TOP_PCT, restrict_mask=lesion)
    ur = ulas_with_null(u, error, lesion, n_permutations=N_PERMS_ULAS, seed=42 + idx)
    rows[g] = dict(cov_g=cov_g, cov_l=cov_l, w_g=w_g, w_l=w_l,
                   iou_g=iou_g, iou_l=iou_l,
                   ulas=ur["ulas"], ulas_null=ur["null_mean"], ulas_z=ur["z_score"])

labels = {"A": "A (ResM)", "B": "B (QR)", "C": "C (QR-Lesion)"}
metric_rows = [
    ("Coverage global",      "cov_g",     "{:.3f}"),
    ("Coverage lesao",       "cov_l",     "{:.3f}"),
    ("Largura media global", "w_g",       "{:.4f}"),
    ("Largura media lesao",  "w_l",       "{:.4f}"),
    ("IoU top-5% global",    "iou_g",     "{:.3f}"),
    ("IoU top-5% lesao",     "iou_l",     "{:.3f}"),
    ("ULAS lesao",           "ulas",      "{:.3f}"),
    ("ULAS null baseline",   "ulas_null", "{:.3f}"),
    ("ULAS z-score",         "ulas_z",    "{:.2f}"),
]
W = 18
print("=" * (26 + 3 * W))
print(f"METRICAS DO SLICE  (vol={sample['volume_id']}, slice={sample['slice_idx']}, "
      f"seq={sample['sequence']}, alpha={ALPHA})")
print("=" * (26 + 3 * W))
hdr = f"{'Metrica':<26}" + "".join(f"{labels[g]:<{W}}" for g in "ABC")
print(hdr)
print("-" * (26 + 3 * W))
for name, key, fmt in metric_rows:
    line = f"{name:<26}"
    for g in "ABC":
        v = rows[g][key]
        line += f"{(fmt.format(v) if v == v else 'nan'):<{W}}"  # v==v: NaN-safe
    print(line)
print("=" * (26 + 3 * W))
print("\nLeitura rapida (este e UM slice; os resultados do TCC sao agregados no S5.8/S5.9):")
print("  - Coverage global ~0.90 nos 3 grupos: garantia formal de Romano et al. (2019).")
print("  - ULAS > null baseline (z alto) indica alinhamento direcional informativo.")


In [ ]:
# Celula 8 — Hero figure: GT | reconstrucao | erro | ResM | QR | QR-Lesion
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif", "font.size": 12,
    "axes.titlesize": 13, "figure.dpi": 120,
    "savefig.dpi": 300, "savefig.bbox": "tight",
})


def to2d(t):
    return t.squeeze().detach().cpu().numpy()


def bboxes_from_mask(mask2d):
    '''Retorna lista de (x, y, w, h) por componente conexa (4-conectividade).
    Usa scipy se disponivel; senao, bbox unico da mascara inteira.'''
    m = mask2d > 0.5
    if not m.any():
        return []
    try:
        from scipy import ndimage
        lab, n = ndimage.label(m)
        boxes = []
        for k in range(1, n + 1):
            ys, xs = np.where(lab == k)
            boxes.append((xs.min(), ys.min(),
                          xs.max() - xs.min() + 1, ys.max() - ys.min() + 1))
        return boxes
    except Exception:
        ys, xs = np.where(m)
        return [(xs.min(), ys.min(), xs.max() - xs.min() + 1, ys.max() - ys.min() + 1)]


gt = to2d(target)
rc = to2d(recon)
er = to2d(error)
maps = {g: to2d(pred[g][3]) for g in "ABC"}   # halfwidth calibrado
mask2d = to2d(lesion)
boxes = bboxes_from_mask(mask2d)

# Escala de cor comum para os 3 mapas de incerteza (comparacao justa)
umax = max(np.percentile(maps[g], 99.5) for g in "ABC")

panels = [
    ("Ground Truth", gt, "gray", None),
    ("Reconstrucao 4x", rc, "gray", None),
    ("Erro |y - x|", er, "inferno", None),
    ("ResM (A)", maps["A"], "inferno", umax),
    ("QR (B)", maps["B"], "inferno", umax),
    ("QR-Lesion (C)", maps["C"], "inferno", umax),
]

fig, axes = plt.subplots(1, 6, figsize=(22, 4))
for ax, (title, img, cmap, vmax) in zip(axes, panels):
    im = ax.imshow(img, cmap=cmap, vmax=vmax)
    ax.set_title(title, fontweight="bold")
    ax.axis("off")
    for (x, y, w, h) in boxes:
        ax.add_patch(plt.Rectangle((x, y), w, h, linewidth=1.6,
                                   edgecolor="lime", facecolor="none"))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    f"{sample['volume_id']}  |  slice {sample['slice_idx']}  |  {sample['sequence']}  "
    f"|  cobertura nominal {int((1 - ALPHA) * 100)}%   "
    f"(incerteza = meia-largura do intervalo calibrado)",
    y=1.06, fontsize=12,
)
plt.tight_layout()

out_dir = Path("figures") if Path("figures").is_dir() else Path(".")
out_path = out_dir / f"hero_demo_{sample['volume_id']}_s{sample['slice_idx']}.png"
plt.savefig(out_path)
print("Hero figure salva em:", out_path)
plt.show()


## Interpretacao (leia antes de mostrar para a banca)

- Este notebook ilustra **um slice**. Os resultados do TCC sao **agregados** sobre 47 volumes de teste no S5.8/S5.9 (Wilcoxon + Holm-Bonferroni, Clopper-Pearson, BCa bootstrap). Nao tire conclusoes de um slice isolado.
- A vantagem do **Grupo A em Coverage_lesion** observada no agregado e um **mecanismo de calibracao** (escala localmente adaptativa do Scaled-CP), **nao** uma diferenca de qualidade de arquitetura. A CQR normalizada fecha e supera essa diferenca (ver `docs/S5.md`).
- O efeito da **loss ponderada por lesao (Grupo C)** e **pequeno mas confiavel** (d_z ~0.19–0.26); o efeito de **calibracao** e grande (d ~0.93). Mantenha o escopo das afirmacoes coerente com isso.
- A coluna de incerteza da hero figure usa a **meia-largura do intervalo calibrado** (`(upper_cal - lower_cal)/2`), comparavel entre os 3 grupos. As metricas de IoU/ULAS usam a incerteza pre-calibracao `u`, exatamente como em `scripts/compute_metrics.py`.

## Tornar o demo publico (bloco 3.2)

Hoje o notebook roda em Kaggle com os datasets privados anexados. Para que **qualquer pessoa** (a banca, por exemplo) consiga clicar e rodar sem acesso ao Kaggle:

1. Rode o bloco 3.2 para publicar no HuggingFace Hub **apenas os `state_dict`** dos 3 modulos + os 3 `q_hat_*.json` + 1 volume de amostra (sem dados de paciente alem do volume publico de teste).
2. Preencha `HF_MODEL_REPO` / `HF_SAMPLE_REPO` na **Celula 2**.
3. O notebook passa a baixar tudo do Hub automaticamente (prioridade 1 na Celula 3).
